In [ ]:
# Empty for "ruff" extension

# Batch and Online algorithms for Near-Duplicates Detection


In [ ]:
import sys
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from pathlib import Path
import pandas as pd
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from src.datasets import read_c5
from src.fingerprint import create_fingerprints


sys.path.append(str(Path.cwd().parent))

from src.spark_utils import get_spark
from src.datasets import read_synthetic

spark = get_spark()
spark.conf.set("spark.sql.debug.maxToStringFields", 10000)
spark.conf.set("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")

%load_ext autoreload
%autoreload 2

In [ ]:
from src.near_duplicates import tables_schema
import math


f = 64
p_min = 10
d = 20

for K in range(2, int(f // 2)):
    tables = tables_schema(f=f, K=K, d=d,p_min=p_min)
    print(K, tables, math.comb(tables[0], tables[0] - K))

In [ ]:
from src.datasets import read_c5
from src.fingerprint import create_fingerprints
from src.near_duplicates import detect_near_duplicate_batch


K = 3
ID_COL = "ID"
FP_COL = "fingerprint"

corpus = read_synthetic()
query = corpus.sample(0.2)
result = detect_near_duplicate_batch(corpus, query, fingerprint_size=64, K=K)
result.count()
spark.sparkContext.show_profiles()

In [ ]:
duplicate_count = df_dup.select("id1").distinct().count()
print(f"Near-Duplicate elements with hamming distance {K}: {duplicate_count} out of {df_size} ({duplicate_count / df_size * 100}%)")

In [ ]:
near_dup_sample = df_dup.filter(F.col("hamming_distance") > 0).limit(5)
near_dup_sample = df.alias("a").join(F.broadcast(near_dup_sample).alias("b"), F.col(f"a.{ID_COL}") == F.col("b.id1")).select("b.id1", "b.id2", F.col("data").alias("text1"), "hamming_distance")
near_dup_sample = df.alias("a").join(F.broadcast(near_dup_sample).alias("b"), F.col(f"a.{ID_COL}") == F.col("b.id2")).select("b.id1", "b.id2", "text1", F.col("data").alias("text2"), "hamming_distance")
near_dup_sample.show(truncate=False)

In [ ]:
df_dup.unpersist()
spark.stop()